# Mutation → Mechanism → Therapy — Main Run Notebook

**Repo:** [github.com/lightflow16/mutation-mechanism-therapy-amd](https://github.com/lightflow16/mutation-mechanism-therapy-amd)

**Clone once** — run the Python clone cell below (section 0). Do **not** paste `git clone ...` as bare Python; that causes `SyntaxError`.

AMD notebook: start in `/workspace`. Colab: start in `/content`.

## 0 — Clone + confirm repo root

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/lightflow16/mutation-mechanism-therapy-amd.git"
REPO_NAME = "mutation-mechanism-therapy-amd"


def _safe_cwd() -> Path:
    """Colab kernels sometimes have a deleted cwd; Path('.').resolve() then fails."""
    try:
        return Path.cwd()
    except OSError:
        for fallback in (Path("/content"), Path("/workspace"), Path.home()):
            if fallback.is_dir():
                os.chdir(fallback)
                return fallback
        os.chdir("/")
        return Path("/")


def _find_repo() -> Path | None:
    candidates = [
        _safe_cwd(),
        Path("/content") / REPO_NAME,
        Path("/workspace") / REPO_NAME,
        _safe_cwd() / REPO_NAME,
    ]
    seen: set[str] = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / "src" / "pipeline.py").is_file():
            return base
    return None


found = _find_repo()
if found is not None:
    os.chdir(found)
    print("Repo root:", found)
else:
    parent = Path("/content") if Path("/content").is_dir() else _safe_cwd()
    dest = parent / REPO_NAME
    if dest.exists():
        import shutil
        shutil.rmtree(dest)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(dest)],
        check=True,
        cwd=str(parent),
    )
    os.chdir(dest)
    print("Cloned to:", dest)

In [ ]:
# Confirm repo root (safe on Colab when cwd was deleted)
from pathlib import Path
import os

ROOT = Path.cwd()
assert (ROOT / "src" / "pipeline.py").is_file(), (
    f"Not in repo root ({ROOT}). Re-run the cell above."
)
print("ROOT =", ROOT)
print("Commit:", end=" ")
!git -C {ROOT} log -1 --oneline

## 1 — Environment + paths (CPU ok, no GPU yet)

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('.').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import configure_paths, is_rocm, setup_env
from src.serving import verify_gpu_torch

configure_paths()
setup_env()

import torch
print('torch', torch.__version__)
print('platform:', 'ROCm/AMD' if is_rocm() else 'CUDA (Colab) or local')
print('LLM_BACKEND', os.environ.get('LLM_BACKEND', 'auto'))
print('GPU verify (CPU ok if not attached yet):', verify_gpu_torch())

In [ ]:
!pip install -q -r requirements.txt openai

In [ ]:
!bash scripts/setup_external.sh

In [ ]:
# ── HuggingFace token + model-cache check ─────────────────────────────────────
# Models used: Qwen/Qwen2.5-VL-7B-Instruct (~15 GB), Qwen/Qwen2.5-7B-Instruct (~15 GB),
#              facebook/esmfold_v1 (~690 MB).
# All are public. A token avoids HF rate-limits and is required if org policies change.
#
# Token resolution order (first found wins):
#   1. HF_TOKEN env var (set in AMD notebook secrets or `export HF_TOKEN=hf_...` in terminal)
#   2. /workspace/shared/hf_token  (drop your token here on the AMD machine)
#   3. ~/.cache/huggingface/token  (written by a prior `huggingface-cli login`)
#   4. No token — proceeds unauthenticated (fine for public models if cache is warm)

import os
from pathlib import Path

_TOKEN_PATHS = [
    Path("/workspace/shared/hf_token"),
    Path.home() / ".cache" / "huggingface" / "token",
]

token = os.environ.get("HF_TOKEN", "").strip()
if not token:
    for _p in _TOKEN_PATHS:
        if _p.is_file():
            token = _p.read_text().strip()
            if token:
                print(f"HF token loaded from {_p}")
                break

if token:
    os.environ["HF_TOKEN"] = token
    try:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        print("huggingface_hub: logged in OK")
    except Exception as e:
        print(f"huggingface_hub login warning (non-fatal): {e}")
else:
    print("No HF_TOKEN found — proceeding unauthenticated (public models only).")
    print("If downloads fail, set HF_TOKEN env var or write token to /workspace/shared/hf_token")

# ── Model cache status ─────────────────────────────────────────────────────────
HF_HOME = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
_MODELS = [
    "Qwen/Qwen2.5-VL-7B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "facebook/esmfold_v1",
]
print(f"\nHF_HOME: {HF_HOME}")
for mid in _MODELS:
    cache_slug = "models--" + mid.replace("/", "--")
    cached = (HF_HOME / "hub" / cache_slug).is_dir()
    status = "CACHED" if cached else "not cached (will download on first use)"
    print(f"  {mid}: {status}")

## 2 — Instant demo (cached traces, NO GPU, NO vLLM)

In [ ]:
from src.pipeline import run_case
import json

for gene, mut in [('EGFR','L858R'), ('PIK3CA','E545K'), ('TP53','R175H')]:
    r = run_case(gene, mut, architecture='blackboard', use_cached_trace=True, live_evidence=False)
    tr = r['reasoning'].get('target_reasoning', r['reasoning'])
    print(gene, mut, '->', r['route'], '| therapies:', tr.get('therapy',{}).get('sensitivity'))

## 2b — MTB panel (cached blackboard trace, CPU only)

Prints the Molecular Tumor Board view: each agent's stance from `trace_*_blackboard.json`.

In [ ]:
from src.config import metrics_dir
from src.mtb_panel import format_mtb_panel
from pathlib import Path

# Prefer live metrics dir; fall back to repo cached trace
candidates = [
    metrics_dir() / 'trace_PIK3CA_E545K_blackboard.json',
    Path('data/traces/PIK3CA_E545K_blackboard.json'),
]
for p in candidates:
    if p.is_file():
        print(format_mtb_panel(p))
        break
else:
    print('No PIK3CA blackboard trace found — run section 2 or 3 first.')

## 2c — Reasoning ablation table (CPU, from saved metrics)

In [ ]:
import pandas as pd
from pathlib import Path
from src.config import metrics_dir

md = metrics_dir()
abl = md / 'ablation_results.csv'
ror = md / 'return_on_reasoning.csv'
ext = md / 'extended_thinking_ablation.csv'
if abl.is_file():
    display(pd.read_csv(abl))
else:
    print('No ablation_results.csv — run eval or full submission first.')
if ror.is_file():
    print('\nReturn on Reasoning:')
    display(pd.read_csv(ror))
if ext.is_file():
    print('\nExtended thinking (thinking_tokens, rubric, conflict rate):')
    display(pd.read_csv(ext))
else:
    from src.extended_thinking_ablation import write_reports
    write_reports()
    if (md / 'extended_thinking_ablation.csv').is_file():
        display(pd.read_csv(md / 'extended_thinking_ablation.csv'))
llm = md / 'llm_calls.jsonl'
if llm.is_file():
    rows = [__import__('json').loads(l) for l in llm.read_text().strip().split('\n') if l]
    df = pd.DataFrame(rows)
    if 'multimodal_image' in df.columns:
        print('\nMultimodal calls:', df.groupby(['architecture','multimodal_image']).size())
    tok_col = 'thinking_tokens' if 'thinking_tokens' in df.columns else 'reasoning_tokens'
    print('\nToken rollup by architecture:')
    display(df.groupby('architecture')[['ingress_tokens','egress_tokens', tok_col]].sum())

## 2d — Unknown / VUS demo (EGFR G719S — abstention path)

In [ ]:
from src.config import load_config, get_target
from src.pipeline import run_case
from src.variant_router import route_variant
from src.evidence import gather_evidence
from src.mtb_panel import format_vus_summary

cfg = load_config()
gene, mut = cfg['pipeline']['vus_demo_cases'][0]
target = get_target(cfg, gene, mut)
# CPU cached replay; set live_evidence=True for one GPU VUS run
r = run_case(gene, mut, architecture=cfg['pipeline'].get('vus_demo_architecture', 'single'),
             use_cached_trace=True, live_evidence=False)
routing = route_variant(target, r.get('structure', {}), r.get('evidence', []))
print('Routing:', routing)
tr = r['reasoning'] if isinstance(r.get('reasoning'), dict) else {}
print(format_vus_summary(tr, routing))

## 3 — Full GPU run (all flows, always)

One command runs **everything** on Colab or AMD:
- cached baseline + **live** single / cot / blackboard on all 3 demo mutations
- per-mutation architecture comparison + eval
- full TP53 rescue (ThermoMPNN + MPNN + ESMFold + Boltz)
- exports **all metrics** (calls.csv, phases.csv, llm_calls.jsonl, system_samples.csv, traces, comparisons)

**AMD:** transformers only (never `pip install vllm`). **Colab:** attach GPU before this section.

In [ ]:
import time
GPU_ATTACH_T0 = time.time()  # budget proxy (~240 min)
print('GPU attached at', GPU_ATTACH_T0)

In [ ]:
from src.config import is_rocm
from src.serving import platform_summary, verify_gpu_torch
from src.llm_client import use_vllm

print(platform_summary())
gpu = verify_gpu_torch()
assert gpu["ok"], f"GPU broken: {gpu.get('error')} — restart session if you ran pip install vllm on AMD"
print("LLM routing:", "vllm" if use_vllm() else "transformers")

In [ ]:
!bash scripts/setup_boltz_venv.sh
import os
from pathlib import Path
boltz = Path('external/boltz_venv/bin/boltz')
assert boltz.is_file(), 'Boltz missing — run setup_external.sh'
os.environ['BOLTZ_BIN'] = str(boltz.resolve())
print('BOLTZ_BIN', os.environ['BOLTZ_BIN'])

## 3b — Teacher–student LoRA (distill blackboard traces → train → infer)

In [ ]:
# Run AFTER blackboard traces exist (section 3 or data/traces/)
!PYTHONPATH=. python train/build_dataset.py --from-traces
!PYTHONPATH=. python train/lora_sft.py
print('LoRA adapter ready at /workspace/shared/lora_adapter_final')

In [ ]:
# Optional retrain (OFF by default): Section 21 already trained LoRA once.
# Set FORCE_RETRAIN=True only if you intentionally want to overwrite adapter weights.
from pathlib import Path

FORCE_RETRAIN = False
adapter_dir = Path('/workspace/shared/lora_adapter_final')
if not adapter_dir.is_dir():
    adapter_dir = Path('.').resolve() / 'shared' / 'lora_adapter_final'
weights_exist = adapter_dir.is_dir() and (
    any(adapter_dir.glob('adapter_model.*')) or any(adapter_dir.glob('*.safetensors'))
)

print('Adapter dir:', adapter_dir)
print('weights_exist:', weights_exist)

if FORCE_RETRAIN or not weights_exist:
    print('Running LoRA training...')
    !pip install -q 'torchao>=0.16.0'
    !PYTHONPATH=. python train/lora_sft.py
else:
    print('Skipping retrain (already trained in section 21).')

In [ ]:
# Pull latest fixes before full run, then flush stale src modules
import subprocess, sys

result = subprocess.run(["git", "pull"], capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

for _m in list(sys.modules):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

print("src modules flushed — will reimport from latest code.")

In [ ]:
# FULL SUBMISSION — all architectures × all cases + eval + metrics bundle
import os
import sys
from pathlib import Path

# Fresh src after git pull (fixes stale metrics.log_llm_call signature on Colab)
for _m in list(sys.modules):
    if _m == 'src' or _m.startswith('src.'):
        del sys.modules[_m]

from src.submission import run_full_submission

LORA = os.environ.get('LORA_PATH', '/workspace/shared/lora_adapter_final')
if not os.path.isdir(LORA):
    alt = Path('.').resolve() / 'shared' / 'lora_adapter_final'
    LORA = str(alt) if alt.is_dir() else None

def _lora_ok(path):
    if not path: return False
    p = Path(path)
    return p.is_dir() and (any(p.glob('adapter_model.*')) or any(p.glob('*.safetensors')))

print('LoRA path:', LORA, '| weights found:', _lora_ok(LORA))
print('Live console echo enabled (pipeline.verbose_progress in targets.yaml)')

report = run_full_submission(lora_path=LORA)
print('Metrics bundle:', report.get('metrics_bundle'))

In [ ]:
## 3c — LoRA comparison matrix results
# Base model vs fine-tuned × single / cot / blackboard / debate
# Populated automatically by the full submission above when a LoRA adapter is loaded.
# Re-run standalone: !PYTHONPATH=. python scripts/run_lora_comparison.py

import pandas as pd
from pathlib import Path
from src.config import metrics_dir

md = metrics_dir()
cmp_csv  = md / "lora_comparison.csv"
cmp_txt  = md / "lora_comparison_summary.txt"

if not cmp_csv.is_file():
    print("No lora_comparison.csv yet — run the full submission cell above (Cell 23).")
    print("The comparison runs automatically when a LoRA adapter is found.")
else:
    # ── summary table ──────────────────────────────────────────────────────────
    if cmp_txt.is_file():
        print(cmp_txt.read_text())

    # ── delta dataframe ────────────────────────────────────────────────────────
    df = pd.read_csv(cmp_csv)

    # Colour helper for notebooks that support pandas Styler
    def _colour(val):
        if pd.isna(val):
            return ""
        return "color: green; font-weight: bold" if val > 0 else (
               "color: red" if val < 0 else "")

    delta_cols = [c for c in df.columns if c.startswith("delta_")]
    styled = (
        df.style
          .applymap(_colour, subset=delta_cols)
          .format({c: "{:+.3f}" for c in delta_cols if c in df.columns}, na_rep="N/A")
          .set_caption("LoRA Fine-tuned vs Base Model — Therapy F1 & Direction Accuracy (Δ = lora − base)")
    )
    display(styled)

    # ── aggregate summary ──────────────────────────────────────────────────────
    print("\nAggregate means across all cells:")
    agg = df[["base_therapy_f1", "lora_therapy_f1", "delta_therapy_f1",
              "base_direction_acc", "lora_direction_acc", "delta_direction_acc"]].mean()
    display(agg.to_frame(name="mean").T.style.format("{:.3f}"))

    # ── per-architecture pivot ─────────────────────────────────────────────────
    print("\nMean Therapy F1 by architecture:")
    pivot = df.pivot_table(
        index="architecture",
        values=["base_therapy_f1", "lora_therapy_f1", "delta_therapy_f1"],
        aggfunc="mean",
    ).round(3)
    display(pivot)

In [ ]:
# (Optional) CLI equivalent:
# !PYTHONPATH=. python scripts/run_full_submission.py --lora-path /workspace/shared/lora_adapter_final

In [ ]:
# TP53 rescue is included in full submission (run_mutation_comparison).
# Uncomment only for an isolated rescue debug run:
# !pip install -q pytorch-lightning torchmetrics omegaconf wandb tqdm
# from src.pipeline import run_mutation_comparison
# run_mutation_comparison('TP53', 'R175H', use_cached_trace=False, run_rescue_branch=True)

## 5 — Multimodal ablation (from llm_calls.jsonl)

In [ ]:
import pandas as pd, json
from src.config import metrics_dir
llm = metrics_dir() / 'llm_calls.jsonl'
if llm.is_file():
    rows = [json.loads(l) for l in llm.read_text().splitlines() if l.strip()]
    df = pd.DataFrame(rows)
    if 'multimodal_image' in df.columns:
        print('Multimodal ablation:')
        display(df.groupby(['architecture','multimodal_image'])[['total_tokens','latency_s']].mean())
else:
    print('Run full submission first')

## 6 — Agent autonomy report (CPU)

In [ ]:
import json
import pandas as pd
from src.config import metrics_dir

md = metrics_dir()
p = md / 'autonomy_report.json'
if not p.is_file():
    print('Run full submission first (trust_eval step)')
else:
    rep = json.loads(p.read_text())
    print('§10 Agent Autonomy (MMT-R)')
    print('  workflow_completion_rate:', rep.get('workflow_completion_rate'))
    print('  task_pass_rate:', rep.get('task_pass_rate'))
    print('  DBTL claim:', rep.get('dbtl_claim'))
    dbtl = rep.get('dbtl_level3_tp53') or {}
    print('  DBTL L3 success:', dbtl.get('dbtl_success'), '| designs:', dbtl.get('n_designs'))
    print('  Refusal rate:', rep.get('refusal_rate'))
    print('  TEVV constraint pass rate:', rep.get('tevv_constraint_pass_rate'))
    print('\nTask suite:')
    display(pd.DataFrame(rep.get('task_suite', [])))
    print('\nABLE metrics:')
    display(pd.DataFrame(rep.get('able_metrics', [])))
    if (md / 'tevv_lite.csv').is_file():
        print('\nTEVV-lite (constraints + refusal):')
        display(pd.read_csv(md / 'tevv_lite.csv'))
    print('\nTrait grid (sample):')
    display(pd.DataFrame(rep.get('trait_grid', [])).head(8))

In [ ]:
# Open workflow dashboard (productive throughput + agent trace timeline)
from IPython.display import IFrame
from src.config import metrics_dir
dash = metrics_dir() / 'workflow_trace_dashboard.html'
if dash.is_file():
    display(IFrame(src=str(dash), width='100%', height=520))
else:
    print('Run full submission first, or: PYTHONPATH=. python scripts/generate_workflow_report.py')

## 4 — Download metrics + Colab vs AMD comparison

In [ ]:
# Place the downloaded Colab metrics bundle so the comparison script can find it.
# Edit COLAB_BUNDLE below to the path where you uploaded / scp-ed the Colab bundle folder.
#
# Expected layout after this cell:
#   ../backups/colab_artifacts_<date>/metrics_bundle_colab_cuda_<timestamp>/  (folder)
# OR a .tgz of the same name (will be auto-extracted below).

import shutil, tarfile
from pathlib import Path

# ── edit this ──────────────────────────────────────────────────────────────────
COLAB_BUNDLE = Path("/workspace/shared/mutation-mechanism-therapy-amd/uploads/metrics_bundle_colab_cuda_20260617_121238")
# ───────────────────────────────────────────────────────────────────────────────

BACKUPS = Path("../backups")

if COLAB_BUNDLE.suffix == ".tgz":
    dest = BACKUPS / "colab_artifacts_20260617" / COLAB_BUNDLE.stem
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(COLAB_BUNDLE) as tf:
        tf.extractall(dest)
    print(f"Extracted Colab bundle → {dest}")
elif COLAB_BUNDLE.is_dir():
    dest = BACKUPS / "colab_artifacts_20260617" / COLAB_BUNDLE.name
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(COLAB_BUNDLE, dest)
    print(f"Colab bundle ready at {dest}")
else:
    print(f"Colab bundle not found at {COLAB_BUNDLE}")
    print("Upload the metrics_bundle_colab_cuda_* folder/tgz to /workspace/shared/mutation-mechanism-therapy-amd/uploads/ and re-run.")

In [ ]:
# Colab vs local/AMD platform comparison (CPU ok if both bundles exist)
!PYTHONPATH=. python scripts/run_amd_compare_demo.py

from pathlib import Path
import json
from src.config import metrics_dir

pc = metrics_dir() / 'platform_comparison.json'
if pc.is_file():
    display(json.loads(pc.read_text()))
else:
    print('Run full submission or place Colab bundle under ../backups/colab_artifacts_*/metrics_bundle_*')

## 7 — Hallucination metrics (§12)

Rule-based HR metrics on cached/live traces: `HR_design`, `HR_property`, `HR_evidence`, `HR_tool`, `BVR`, `HR_policy`. Outputs join the metrics bundle and the workflow dashboard **Trust layer** card.

In [ ]:
from pathlib import Path
import json
from src.config import metrics_dir, setup_env
from src.hallucination_eval import write_report

setup_env()
write_report()
md = metrics_dir()
summary = json.loads((md / 'hallucination_summary.json').read_text()) if (md / 'hallucination_summary.json').is_file() else {}
print('Hallucination summary:')
for k, v in summary.items():
    if k != 'by_architecture':
        print(f'  {k}: {v}')
if summary.get('by_architecture'):
    print('  by_architecture:', json.dumps(summary['by_architecture'], indent=2))
report = md / 'hallucination_report.csv'
if report.is_file():
    print(report.read_text()[:1200])

## 8 — Fold confidence figures

In [ ]:
!PYTHONPATH=. python scripts/plot_confidence_benchmark.py
from IPython.display import SVG
from src.config import metrics_dir
svg = metrics_dir() / 'confidence_scatter_F1.svg'
if svg.is_file():
    display(SVG(filename=str(svg)))
else:
    print('No benchmark_confidence.csv yet')

In [ ]:
from pathlib import Path
from src.config import metrics_dir, shared_dir, configure_paths
from src.metrics_bundle import export_metrics_bundle

configure_paths()
md = metrics_dir()
shared = shared_dir()
bundle = export_metrics_bundle(shared / 'metrics_bundle_latest.tgz')
print('Bundle:', bundle)
print('Metrics dir:', md)
for name in sorted(md.glob('*')):
    if name.is_file():
        print(' ', name.name, name.stat().st_size)

# Colab: download bundle to laptop
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('On AMD: copy from', bundle)

## 8 — Post-fix re-run: Boltz fold on AMD (DBTL fold gate)

On AMD the DBTL **fold gate failed** because Boltz produced no output: Boltz ships CUDA-only wheels, so its bundled PyTorch Lightning can't drive a ROCm GPU. The old code only retried on CPU when the error contained one exact string (`"No supported gpu backend found"`); ROCm 7.2.4 emits a different message, so the retry never fired and `fold_boltz` silently returned `None` → `fold_method = "esmfold"` → gate **FAIL**.

`src/rescue.py` now folds Boltz **on CPU directly on ROCm hosts** (and falls back to CPU on *any* GPU failure elsewhere). When Boltz runs, `fold_method = "boltz+esmfold"`, so `fold_gate_passed` and `dbtl_success` become **True** — a legitimate pass (Boltz genuinely predicts the structure, just on CPU).

**Self-contained:** the cell monkey-patches `fold_boltz` at runtime, so you do *not* need to edit or copy `src/rescue.py` — just paste and run. It re-folds **only the TP53 rescue** (not the LLM architectures), patches the fresh `rescue` block into the TP53 traces, and regenerates the DBTL / fold-confidence / autonomy reports + bundle. It is **opt-in** (`RUN_RESCUE_RERUN = False`) because CPU Boltz can take many minutes, and only meaningful on the AMD host with the Boltz venv installed (`bash scripts/setup_boltz_venv.sh`).

In [ ]:
# POST-FIX RE-RUN — TP53 rescue: Boltz CPU fold, then refresh DBTL/fold/autonomy reports
RUN_RESCUE_RERUN = False  # set True on the AMD host (boltz venv installed) to re-fold TP53

import os, sys, json
from pathlib import Path

# Fresh src so the new rescue.py CPU-fold path is picked up
for _m in list(sys.modules):
    if _m == 'src' or _m.startswith('src.'):
        del sys.modules[_m]

from src.config import (
    load_config, metrics_dir, setup_env, shared_dir, configure_paths, get_target, is_rocm,
)

setup_env()
configure_paths()
cfg = load_config()
md = metrics_dir()
ROOT = Path('.').resolve()
TRACE_CACHE = ROOT / 'data' / 'traces'
GENE, MUT = 'TP53', 'R175H'

boltz_bin = os.environ.get('BOLTZ_BIN', str(ROOT / 'external' / 'boltz_venv' / 'bin' / 'boltz'))
boltz_ok = Path(boltz_bin).is_file()
print(f'is_rocm={is_rocm()} | boltz_bin={boltz_bin} | installed={boltz_ok}')
if not boltz_ok:
    print('  Boltz venv missing -> run: bash scripts/setup_boltz_venv.sh (required to fold on AMD)')

if not RUN_RESCUE_RERUN:
    print('\nRUN_RESCUE_RERUN=False -> skipping. Set True on the AMD host to re-fold TP53 with Boltz (CPU).')
else:
    import subprocess, shutil
    from src import metrics
    from src.structure import analyze_target
    from src.rescue import run_rescue, interpret_rescue
    import src.rescue as _rescue

    os.environ.setdefault('BOLTZ_CPU_TIMEOUT_S', '3600')  # CPU Boltz is slow; allow up to 1h

    # --- self-contained fix (no src edit needed): Boltz folds on CPU on ROCm ---
    # Boltz ships CUDA-only wheels; on a ROCm host fold on CPU directly, and on
    # other hosts fall back to CPU on ANY GPU failure. Idempotent with src/rescue.py.
    def _patched_fold_boltz(sequence, yaml_path, out_dir, cache_dir, *, required=True):
        boltz_bin = os.environ.get('BOLTZ_BIN', str(_rescue.ROOT / 'external' / 'boltz_venv' / 'bin' / 'boltz'))
        if not Path(boltz_bin).is_file() and not shutil.which(boltz_bin):
            if required:
                raise FileNotFoundError(f'Boltz not found ({boltz_bin}). Run: bash scripts/setup_boltz_venv.sh')
            return None
        yaml_path.parent.mkdir(parents=True, exist_ok=True)
        out_dir.mkdir(parents=True, exist_ok=True)
        cache_dir.mkdir(parents=True, exist_ok=True)
        yaml_path.write_text(
            f'version: 1\nsequences:\n  - protein:\n      id: A\n      sequence: {sequence}\n      msa: empty\n'
        )

        def _cmd(acc):
            c = [boltz_bin, 'predict', str(yaml_path.name), '--out_dir', str(out_dir),
                 '--cache', str(cache_dir), '--accelerator', acc]
            if acc == 'gpu':
                c += ['--devices', '1']
            return c + ['--recycling_steps', '1', '--diffusion_samples', '1',
                        '--no_kernels', '--output_format', 'pdb']

        def _reset():
            shutil.rmtree(str(out_dir), ignore_errors=True)
            out_dir.mkdir(parents=True, exist_ok=True)

        gpu_to = int(os.environ.get('BOLTZ_TIMEOUT_S', '600'))
        cpu_to = int(os.environ.get('BOLTZ_CPU_TIMEOUT_S', '2400'))
        try:
            rocm_host = bool(is_rocm())
        except Exception:
            rocm_host = False

        with metrics.track('boltz_fold', agent_role='Rescue', model='boltz-2.2.1'):
            r = None
            if not rocm_host:
                try:
                    r = subprocess.run(_cmd('gpu'), cwd=str(yaml_path.parent),
                                       capture_output=True, text=True, timeout=gpu_to)
                except subprocess.TimeoutExpired:
                    if required:
                        raise RuntimeError(f'Boltz timed out after {gpu_to}s (GPU run)')
                    return None
                except OSError as exc:
                    if required:
                        raise RuntimeError(f'Boltz failed to start: {exc}') from exc
                    return None
            if rocm_host or r is None or r.returncode != 0:
                _reset()
                try:
                    r = subprocess.run(_cmd('cpu'), cwd=str(yaml_path.parent),
                                       capture_output=True, text=True, timeout=cpu_to)
                except subprocess.TimeoutExpired:
                    if required:
                        raise RuntimeError(f'Boltz timed out after {cpu_to}s (CPU fold)')
                    return None
                except OSError as exc:
                    if required:
                        raise RuntimeError(f'Boltz failed to start on CPU: {exc}') from exc
                    return None
            if r.returncode != 0:
                detail = (r.stderr or r.stdout or '').strip()
                if required:
                    raise RuntimeError(f'Boltz failed (exit {r.returncode}):\n{detail[-4000:]}')
                return None
        pdbs = list(out_dir.rglob('*.pdb'))
        if not pdbs:
            if required:
                raise RuntimeError(f'Boltz produced no PDB under {out_dir}')
            return None
        return pdbs[0]

    _rescue.fold_boltz = _patched_fold_boltz
    print(f'Patched fold_boltz (CPU fold on ROCm). is_rocm={is_rocm()}')
    # --- end fix ---

    target = get_target(cfg, GENE, MUT)
    structure = analyze_target(target, shared_dir(cfg) / 'structures')
    print(f'Re-folding {GENE} {MUT} rescue (CPU Boltz on ROCm)...')
    rescue = run_rescue(target, Path(structure['pdb_path']))
    rescue['interpreter'] = interpret_rescue(target, rescue)
    print(f"  fold_method={rescue.get('fold_method')} | boltz_pdb={rescue.get('boltz_pdb')}")

    # Patch the fresh rescue block into every TP53 trace (both caches)
    for arch in list(cfg['pipeline'].get('architectures', ['single', 'cot', 'blackboard'])):
        for p in (md / f'trace_{GENE}_{MUT}_{arch}.json', TRACE_CACHE / f'{GENE}_{MUT}_{arch}.json'):
            if p.exists():
                t = json.loads(p.read_text())
                t['rescue'] = rescue
                p.write_text(json.dumps(t, indent=2, default=str))

    # Regenerate DBTL / fold-confidence / autonomy + platform summary + bundle
    from src.agent_autonomy_eval import run as run_autonomy
    from src.fold_confidence_eval import write_benchmark as write_fold_confidence
    from src.metrics_bundle import write_platform_summary, export_metrics_bundle

    write_fold_confidence()
    run_autonomy(live_probes=bool(cfg['pipeline'].get('live_safety_probes', False)))
    metrics.aggregate_ablation()
    write_platform_summary()
    bundle = export_metrics_bundle(shared_dir() / 'metrics_bundle_latest.tgz')
    print('Refreshed bundle:', bundle)

    dbtl_path = md / 'dbtl_metrics.json'
    if dbtl_path.is_file():
        d = json.loads(dbtl_path.read_text())
        print(f"\nDBTL after re-fold: fold_method={d.get('fold_method')} | "
              f"fold_gate_passed={d.get('fold_gate_passed')} | "
              f"ddg_gate_passed={d.get('ddg_gate_passed')} | "
              f"dbtl_success={d.get('dbtl_success')}")